# Dependency Syntax: Attention Weights vs. Norm-Based Maps

Python 3 / PyTorch version of `Syntax_Analysis.ipynb` (Sections 4.2 and 5 of
[Clark et al., 2019](https://arxiv.org/abs/1906.04341)), run on the attention
weights $\alpha$ (M1) and on the norm-based maps $\|\alpha f(x)\|$ (M2) of
[Kobayashi et al., 2020](https://www.aclweb.org/anthology/2020.emnlp-main.574/).

Data: word-level maps written by `extract_norms.py --word_level`, e.g. for UD
(the paper used the WSJ Penn Treebank with Stanford Dependencies, which is not
freely available):
```
python preprocess_conllu.py --conllu en_ewt-ud-train.conllu --outfile $DATA/train.json
python preprocess_conllu.py --conllu en_ewt-ud-dev.conllu --outfile $DATA/dev.json
python extract_norms.py --preprocessed-data-file $DATA/dev.json --bert-dir bert-base-uncased --word_level
python extract_norms.py --preprocessed-data-file $DATA/train.json --bert-dir bert-base-uncased --word_level
# control: same model architecture, random weights
python extract_norms.py --preprocessed-data-file $DATA/dev.json --bert-dir bert-base-uncased --word_level \
    --random_init --outfile $DATA/dev_random_norms.pkl
```
Note that examples longer than `--max_sequence_length` tokens are dropped, as in
the original code.

In [ ]:
import collections
import os

import numpy as np
import torch
from matplotlib import pyplot as plt

import analysis_utils as au
import utils

DATA_DIR = os.environ.get("ATTN_DATA_DIR", "./data")
DEV_FILE = os.path.join(DATA_DIR, "dev_norms.pkl")
TRAIN_FILE = os.path.join(DATA_DIR, "train_norms.pkl")
RANDOM_DEV_FILE = os.path.join(DATA_DIR, "dev_random_norms.pkl")  # optional
GLOVE_DIR = os.path.join(DATA_DIR, "glove")  # optional, from the paper's data

In [ ]:
# list of dicts with "words", "heads" (0 = ROOT, 1 = first word), "relns",
# "attns" and "norms" ([layers, heads, n_words + 2, n_words + 2], including
# [CLS] and [SEP])
dev_data = utils.load_pickle(DEV_FILE)
print(len(dev_data), "dev sentences")
print("words:", dev_data[0]["words"])
print("heads:", dev_data[0]["heads"])
print("relns:", dev_data[0]["relns"])
print("attns:", dev_data[0]["attns"].shape, "norms:", dev_data[0]["norms"].shape)

reln_counts = collections.Counter(r for e in dev_data for r in e["relns"])
print(reln_counts.most_common(10))

### Individual heads (Section 4.2, Table 1)
Each word is assigned the word it attends to most (or, for `h<-d`, the word
that attends to it most), ignoring the diagonal and [CLS]/[SEP]. The argmax is
invariant to rescaling a row, so no normalization is needed here; M2 differs
from M1 only by scaling column $j$ with $\|f(x_j)\|$.

In [ ]:
head_scores = {"attns": au.get_head_scores(dev_data, "attns"),
               "norms": au.get_head_scores(dev_data, "norms")}
if os.path.exists(RANDOM_DEV_FILE):
  random_dev_data = utils.load_pickle(RANDOM_DEV_FILE)
  head_scores["attns (random)"] = au.get_head_scores(random_dev_data, "attns")
  head_scores["norms (random)"] = au.get_head_scores(random_dev_data, "norms")
baseline_scores = au.get_baseline_scores(dev_data)
table = au.best_head_table(dev_data, head_scores, baseline_scores)
au.print_best_head_table(table)

Best head per relation is selected on the same data it is scored on (as in
the paper), so these numbers are optimistic; for a fair comparison select heads
on one split and report on another:

In [ ]:
def held_out_best_heads(select_data, eval_data, key):
  select = au.get_head_scores(select_data, key)
  evaluate = au.get_head_scores(eval_data, key)
  rows = {}
  for reln, count, _, _, best in au.best_head_table(
      eval_data, {key: select}, au.get_baseline_scores(eval_data)):
    _, layer, head, direction = best[key]
    rows[reln] = evaluate[direction][layer][head].get(reln, 0.0)
  return rows

half = len(dev_data) // 2
for key in ["attns", "norms"]:
  held_out = held_out_best_heads(dev_data[:half], dev_data[half:], key)
  print(key, {r: round(100 * a, 1) for r, a in held_out.items()})

### Per-head accuracy, M2 - M1
Difference in accuracy of every head between the norm-based and the attention
maps, for a few relations (both directions, best of the two).

In [ ]:
RELNS = [r for r, _ in reln_counts.most_common() if r not in ("punct", "root")][:6]
fig, axes = plt.subplots(1, len(RELNS), figsize=(3.2 * len(RELNS), 3))
n_layers, n_heads = dev_data[0]["attns"].shape[:2]
for ax, reln in zip(axes, RELNS):
  diff = np.zeros((n_layers, n_heads))
  for l in range(n_layers):
    for h in range(n_heads):
      best = lambda key: max(head_scores[key][d][l][h].get(reln, 0.0)
                             for d in ["dep->head", "head<-dep"])
      diff[l, h] = best("norms") - best("attns")
  lim = max(np.abs(diff).max(), 1e-6)
  im = ax.imshow(100 * diff, cmap="RdBu", vmin=-100 * lim, vmax=100 * lim)
  ax.set_title(reln)
  ax.set_xlabel("head")
  ax.set_ylabel("layer")
  plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### Probing classifiers (Section 5)
PyTorch port of the paper's probes (batch size 1, Adam with lr 0.002, one epoch,
UAS excluding punctuation). The attention-only probe is a linear combination of
the 144 maps and their transposes. Unlike the argmax above, it is sensitive to
the scale of the features, so for M2 we report raw and row-normalized maps.
Results vary with the random seed; report several.

In [ ]:
train_data = utils.load_pickle(TRAIN_FILE)
print(len(train_data), "train sentences")

SEEDS = [0, 1, 2]
probe_results = collections.defaultdict(list)
probes = {}
for key, normalize in [("attns", False), ("norms", False), ("norms", True)]:
  name = key + (" (normalized)" if normalize else "")
  for seed in SEEDS:
    torch.manual_seed(seed)
    probe = au.attn_linear_combo(
        n_maps=int(np.prod(dev_data[0][key].shape[:2])))
    uas = au.run_training(probe, train_data, dev_data, key=key,
                          normalize=normalize, log_every=0)
    probe_results[name].append(uas)
    probes[(name, seed)] = probe
for name, uas in probe_results.items():
  print("{:20s} UAS {:.1f} +- {:.1f}".format(
      name, 100 * np.mean(uas), 100 * np.std(uas)))

In [ ]:
# learned weights of the attention-only probes (seed 0): [dependent->candidate
# maps | candidate->dependent maps], one weight per layer and head
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, probe_results):
  w = probes[(name, 0)].attn_map_weights.detach().numpy()
  w = w.reshape(2, n_layers, n_heads).transpose(1, 0, 2).reshape(n_layers, -1)
  lim = np.abs(w).max()
  im = ax.imshow(w, cmap="RdBu", vmin=-lim, vmax=lim)
  ax.axvline(n_heads - 0.5, color="k")
  ax.set_title(name)
  ax.set_xlabel("head (left: d->h, right: h<-d)")
  ax.set_ylabel("layer")
  plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Probes with GloVe word embeddings need the paper's `glove/embeddings.pkl`
and `glove/vocab.pkl`. See the `fix_root_alignment` note in
`analysis_utils.Probe` about how candidate heads and word embeddings are aligned in
the original code.

In [ ]:
if os.path.exists(os.path.join(GLOVE_DIR, "embeddings.pkl")):
  embeddings = au.WordEmbeddings(os.path.join(GLOVE_DIR, "embeddings.pkl"),
                                 os.path.join(GLOVE_DIR, "vocab.pkl"))
  for key in ["attns", "norms"]:
    torch.manual_seed(0)
    print(key, "attn-and-words")
    au.run_training(au.attn_and_words(embeddings), train_data, dev_data, key=key,
                    log_every=0)
  torch.manual_seed(0)
  print("words-and-distances baseline")
  au.run_training(au.words_and_distances(embeddings), train_data, dev_data,
                  log_every=0)
else:
  print("no GloVe data found in", GLOVE_DIR)